In [28]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/cyrilgarcia
tsnn module path: /Users/cyrilgarcia/tsnn


In [29]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn import tstorch
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
from tsnn.tstorch import transformers
import torch.nn.functional as F
import math
from typing import Optional

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from torch import nn
from tqdm import tqdm
device = 'mps'


plt.style.use('ggplot')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
# This is the new version of the notebook to make all the figures for the paper.

In [31]:
from tsnn.tstorch import models
from tsnn.benchmarks import torch_benchmarks
from tsnn.benchmarks.torch_benchmarks import MSELossWithL1Sparsity

In [32]:
# Notebook to run all the experiments for the paper.

In [33]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def causal_mask_radius(b, h, q_idx, kv_idx):
    return (q_idx >= kv_idx) & (q_idx <= kv_idx+1)

def plot_mask(mask_fn, seq_len=20, title=None, device="cpu"):
    """
    Plot a binary attention mask defined by mask_fn(b,h,q_idx,kv_idx)
    as a (seq_len x seq_len) matrix.
    """
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device) 
    h = torch.zeros(1, device=device)

    mask = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])
    mask = mask.float().cpu()

    return pd.DataFrame(mask).style.background_gradient(axis=None).format(precision=0)

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

In [34]:
def get_ols_corr(N_fea, T_train, rho, oos=True):
    ratio = N_fea/T_train
    if oos:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio/(1-ratio))
    else:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio)


In [35]:
def keep_topk_per_row3(x, k=3):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out

In [36]:
def keep_by_max_value2(x, frac=0.2):
    row_max = x.max(dim=-1, keepdim=True).values
    threshold = frac * row_max
    out = (x >= threshold).to(x.dtype)
    return out

def keep_by_max_value1(x, frac=0.1):
    row_max = x.max(dim=-1, keepdim=True).values
    threshold = frac * row_max
    out = (x >= threshold).to(x.dtype)
    return out


# Run of all models on all effects

In [37]:
def run_models1(rhos, effect, T=4000, n_ts=10, n_f=5, n_rolling=10):

    list_effects1 = [effect]*n_f

    if n_f >=10:
        half_n_f = int(n_f/2)
        correl_split_by_fea1 = [1/np.sqrt(half_n_f)]*(half_n_f) + [0]*half_n_f 
    else:
        correl_split_by_fea1 = [1/np.sqrt(n_f)]*n_f

    mask = causal_mask
    mask_c = build_attention_mask(mask, n_rolling, device=device)
    
    z = generators.Generator(T, n_ts, n_f)
    
    res_train = []
    res_test = []

    for (i,rho) in enumerate(rhos):
        print("Running correl level:", rho)
        theo_correl_is = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=False)
        theo_correl_oos = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=True)


        z.generate_dataset_gr_simple(
            global_corr=rho, 
            correl_split_by_fea=correl_split_by_fea1, 
            list_type_effects=list_effects1,
            list_type_interaction=["cond"], 
            random_ts_shift=n_rolling,
        )
        z.get_dataloader(n_rolling=n_rolling)

        lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                                verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
        lasso_full.fit(z.train)
        boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
        boost_model.fit(z.train)

        comp = benchmark_comparison.Comparator(models=[lasso_full, boost_model], model_names=['lasso_full', 'boosting'])
        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        # Save results
        res_train.append({
            "rho": rhos[i],
            "model": 'Lasso',
            "train_corr_optimal": corr_train.loc['lasso_full', "optimal"]
        })
        res_test.append({
            "rho": rhos[i],
            "model": 'Lasso',
            "test_corr_optimal": corr_test.loc['lasso_full', "optimal"]
        })

        res_train.append({
            "rho": rhos[i],
            "model": 'Boosting',
            "train_corr_optimal": corr_train.loc['boosting', "optimal"]
        })
        res_test.append({
            "rho": rhos[i],
            "model": 'Boosting',
            "test_corr_optimal": corr_test.loc['boosting', "optimal"]
        })

        models_torch = {                 
        'Global_MLP': models.GlobalMLP(n_ts, n_f, n_rolling, dropout=0.1).to(device),          
        # '2D_Trans_TC': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        # ).to(device),
        '2D_Trans_TCTC': models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None, roll_y=True, embeddings="both",
        ).to(device),      
        # '2D_Trans_TC_sparse': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        # ).to(device),
        # '2D_Trans_TCTC_sparse1': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_topk_per_row3, roll_y=True, embeddings="both",
        # ).to(device),
        #  '2D_Trans_TCTC_spar_max01': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_by_max_value1, roll_y=True, embeddings="both",
        # ).to(device),
        #  '2D_Trans_TCTC_spar_max02': models.CustomBiDimensionalTransformer(
        #     n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
        #     dropout=0.1, d_model=64, dim_feedforward=256, sparsify=keep_by_max_value2, roll_y=True, embeddings="both",
        # ).to(device),
        }

        models_torch = {
            k: torch_benchmarks.TorchWrapper(
                models_torch[k], 
                optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
                loss_fn=nn.MSELoss()  
            ) for k in models_torch
        }

        
        for k in models_torch:
            roll_y = True
            if k in ['Global_MLP']:
                roll_y = False
            z.get_dataloader(n_rolling=n_rolling, roll_y=roll_y)
            epochs=40
            models_torch[k].fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)
            
      
        comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
        model_names=[k for k in models_torch]
        )

        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        for k in models_torch:

            train_corr = corr_train.loc[k, "optimal"]
            test_corr  = corr_test.loc[k, "optimal"]

            res_train.append({
                "rho": rhos[i],
                "model": k,
                "train_corr_optimal": train_corr
            })
            res_test.append({
                "rho": rhos[i],
                "model": k,
                "test_corr_optimal": test_corr
            })


        res_train.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "train_corr_optimal": theo_correl_is
            })
        res_test.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "test_corr_optimal": theo_correl_oos
            })

    
    res_train = pd.DataFrame(res_train)
    res_test  = pd.DataFrame(res_test)

    res_train = res_train.pivot(index="rho", columns="model", values="train_corr_optimal")
    res_test  = res_test.pivot(index="rho", columns="model", values="test_corr_optimal")

    return res_train, res_test

## n_f = 5

In [11]:
all_effects_nf5_train_dic = {}
all_effects_nf5_test_dic = {}

In [12]:
#for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
# for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
for effect in ['TS_shift']:
    print("Running_effect:", effect)
    # out_train1, out_test1 = run_models1([0.02, 0.05, 0.1, 0.2, 0.5], effect, T=4000, n_ts=10, n_f=5, n_rolling=10)
    out_train1, out_test1 = run_models1([0.02, 0.1], effect, T=4000, n_ts=10, n_f=5, n_rolling=10)
    all_effects_nf5_train_dic[effect] = out_train1
    all_effects_nf5_test_dic[effect] = out_test1

Running_effect: TS_shift
Running correl level: 0.02
Running correl level: 0.1


In [13]:
# for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
for effect in ['TS_shift']:
    print(effect)
    display(all_effects_nf5_train_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
    display(all_effects_nf5_test_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

TS_shift


rho,0.020000,0.100000
model,,
2D_Trans_TCTC,0.026,0.120
Boosting,0.056,0.286
Global_MLP,0.029,0.110
Lasso,0.279,0.934
theo_correl,0.045,0.219


rho,0.020000,0.100000
model,,
2D_Trans_TCTC,0.016,0.103
Boosting,0.064,0.374
Global_MLP,0.043,0.172
Lasso,0.279,0.932
theo_correl,0.040,0.197


In [14]:
all_effects_nf5_train_dic = {}
all_effects_nf5_test_dic = {}

In [15]:
#for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
# for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
for effect in ['CS_shift']:
    print("Running_effect:", effect)
    # out_train1, out_test1 = run_models1([0.02, 0.05, 0.1, 0.2, 0.5], effect, T=4000, n_ts=10, n_f=5, n_rolling=10)
    out_train1, out_test1 = run_models1([0.02, 0.1], effect, T=4000, n_ts=10, n_f=5, n_rolling=10)
    all_effects_nf5_train_dic[effect] = out_train1
    all_effects_nf5_test_dic[effect] = out_test1

Running_effect: CS_shift
Running correl level: 0.02
Running correl level: 0.1


In [16]:
# for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
for effect in ['CS_shift']:
    print(effect)
    display(all_effects_nf5_train_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
    display(all_effects_nf5_test_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

CS_shift


rho,0.020000,0.100000
model,,
2D_Trans_TCTC,0.127,0.278
Boosting,0.018,0.083
Global_MLP,0.033,0.097
Lasso,0.024,0.173
theo_correl,0.045,0.219


rho,0.020000,0.100000
model,,
2D_Trans_TCTC,0.101,0.259
Boosting,-0.005,0.023
Global_MLP,0.058,0.159
Lasso,0.006,0.176
theo_correl,0.040,0.197


In [17]:
#for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift', 'TS_cond', 'CS_cond']:
# for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
for effect in ['TSCS_shift']:
    print("Running_effect:", effect)
    # out_train1, out_test1 = run_models1([0.02, 0.05, 0.1, 0.2, 0.5], effect, T=4000, n_ts=10, n_f=5, n_rolling=10)
    out_train1, out_test1 = run_models1([0.02, 0.1], effect, T=4000, n_ts=10, n_f=5, n_rolling=10)
    all_effects_nf5_train_dic[effect] = out_train1
    all_effects_nf5_test_dic[effect] = out_test1

Running_effect: TSCS_shift
Running correl level: 0.02
Running correl level: 0.1


In [19]:
# for effect in ['lin', 'TS_shift', 'CS_shift', 'fea_cond', 'TSCS_shift']:
for effect in ['TSCS_shift']:
    print(effect)
    display(all_effects_nf5_train_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
    display(all_effects_nf5_test_dic[effect].T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

TSCS_shift


rho,0.020000,0.100000
model,,
2D_Trans_TCTC,0.002,0.044
Boosting,0.018,0.099
Global_MLP,0.021,0.109
Lasso,-0.002,0.186
theo_correl,0.045,0.219


rho,0.020000,0.100000
model,,
2D_Trans_TCTC,-0.001,-0.003
Boosting,0.007,0.047
Global_MLP,0.035,0.151
Lasso,-0.010,0.187
theo_correl,0.040,0.197


## Diagnostic: focus on `TCTC` only

This section reuses the same `run_models1(...)` setup as the notebook above and only keeps the `2D_Trans_TCTC` results.
The goal is to compare `TS_shift`, `CS_shift`, and `TSCS_shift` without changing the training or evaluation pipeline.


In [28]:
def run_tctc_effect_diagnostic(
    effects=('TS_shift', 'CS_shift', 'TSCS_shift'),
    rhos=(0.02, 0.1),
    T=4000,
    n_ts=10,
    n_f=5,
    n_rolling=10,
):
    results = []
    full_train = {}
    full_test = {}

    for effect in effects:
        out_train, out_test = run_models1(
            list(rhos),
            effect,
            T=T,
            n_ts=n_ts,
            n_f=n_f,
            n_rolling=n_rolling,
        )

        full_train[effect] = out_train
        full_test[effect] = out_test

        for rho in rhos:
            results.append({
                'effect': effect,
                'rho': rho,
                'train_corr_optimal': out_train.loc[rho, '2D_Trans_TCTC'],
                'test_corr_optimal': out_test.loc[rho, '2D_Trans_TCTC'],
            })

    return pd.DataFrame(results), full_train, full_test


tctc_diag_results, tctc_diag_train_tables, tctc_diag_test_tables = run_tctc_effect_diagnostic()


Running correl level: 0.02
Running correl level: 0.1
Running correl level: 0.02
Running correl level: 0.1
Running correl level: 0.02
Running correl level: 0.1


In [29]:
display(tctc_diag_results.style.format(precision=3))
display(tctc_diag_results.pivot(index='effect', columns='rho', values='train_corr_optimal').style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(tctc_diag_results.pivot(index='effect', columns='rho', values='test_corr_optimal').style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))


,effect,rho,train_corr_optimal,test_corr_optimal
0,TS_shift,0.020,0.010,0.005
1,TS_shift,0.100,0.150,0.096
2,CS_shift,0.020,0.044,0.056
3,CS_shift,0.100,0.197,0.181
4,TSCS_shift,0.020,0.007,-0.004
5,TSCS_shift,0.100,0.034,-0.006


rho,0.020000,0.100000
effect,,
CS_shift,0.044,0.197
TSCS_shift,0.007,0.034
TS_shift,0.010,0.150


rho,0.020000,0.100000
effect,,
CS_shift,0.056,0.181
TSCS_shift,-0.004,-0.006
TS_shift,0.005,0.096


## Generator-based TCTC diagnostics

This section now reuses the project generator instead of a custom synthetic helper.
We compare `TS_shift`, `CS_shift`, and `TSCS_shift` under the exact same data-generation path as the rest of the notebook.


In [38]:
def _default_corr_split(n_f):
    if n_f >= 10:
        half_n_f = int(n_f / 2)
        return [1 / np.sqrt(half_n_f)] * half_n_f + [0] * half_n_f
    return [1 / np.sqrt(n_f)] * n_f


def generate_project_generator_dataset(
    effect='TSCS_shift',
    T=4000,
    n_ts=10,
    n_f=5,
    rho=0.1,
    n_rolling=10,
    seed=0,
):
    np.random.seed(seed)
    torch.manual_seed(seed)

    z = generators.Generator(T, n_ts, n_f)
    z.generate_dataset_gr_simple(
        global_corr=rho,
        correl_split_by_fea=_default_corr_split(n_f),
        list_type_effects=[effect] * n_f,
        list_type_interaction=['cond'],
        random_ts_shift=n_rolling,
    )

    return z.X, z.y, z.ys['optimal'], z


def build_loaders_from_tensors(X, y, n_rolling=10, batch_size=256, train_pct=0.625):
    dataset = utils.TorchDatasetRolling(X, y, n=n_rolling, roll_y=True)
    train_size = int(train_pct * len(dataset))

    train_data = torch.utils.data.Subset(dataset, range(train_size))
    test_data = torch.utils.data.Subset(dataset, range(train_size, len(dataset)))

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=utils.collate_pad_beginning)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=True, collate_fn=utils.collate_pad_beginning)

    return train_loader, test_loader


def corr_to_optimal(model_wrapper, dataloader, y_opt):
    preds = []
    model_wrapper.model.eval()

    with torch.no_grad():
        for batch in dataloader:
            if len(batch) == 3:
                X, _, pad_mask = batch
                pred = model_wrapper.model(X.to(device), pad_mask=pad_mask.to(device)).detach().cpu()
            else:
                X, _ = batch
                pred = model_wrapper.model(X.to(device)).detach().cpu()

            if pred.ndim == 3:
                pred = pred[:, -1, :]
            preds.append(pred.flatten())

    preds = torch.cat(preds).numpy()
    idx = dataloader.dataset.indices
    target = y_opt[idx].reshape(-1).numpy()

    return np.corrcoef(preds, target)[0][1]


def run_tctc_tscs_ablation(
    rhos=(0.1, 0.3),
    effects=('TS_shift', 'CS_shift', 'TSCS_shift'),
    T=4000,
    n_ts=10,
    n_f=5,
    n_rolling=10,
    epochs=20,
    batch_size=256,
    seed=0,
):
    mask_c = build_attention_mask(causal_mask, n_rolling, device=device)
    results = []

    for effect in effects:
        for rho in rhos:
            X, y, y_opt, _ = generate_project_generator_dataset(
                effect=effect,
                T=T,
                n_ts=n_ts,
                n_f=n_f,
                rho=rho,
                n_rolling=n_rolling,
                seed=seed,
            )

            train_loader, test_loader = build_loaders_from_tensors(
                X, y, n_rolling=n_rolling, batch_size=batch_size
            )

            model = models.CustomBiDimensionalTransformer(
                n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
                dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None,
                roll_y=True, embeddings='both',
            ).to(device)

            wrapper = torch_benchmarks.TorchWrapper(
                model,
                optimizer=torch.optim.AdamW(model.parameters(), lr=0.001),
                loss_fn=nn.MSELoss(),
                device=device,
            )

            wrapper.fit(train_loader, test=test_loader, epochs=epochs, plot=False, verbose=0)

            results.append({
                'effect': effect,
                'rho': rho,
                'train_corr_optimal': corr_to_optimal(wrapper, train_loader, y_opt),
                'test_corr_optimal': corr_to_optimal(wrapper, test_loader, y_opt),
                'last_train_batch_corr': wrapper.train_corr[-1],
                'last_test_batch_corr': wrapper.test_corr[-1],
            })

    return pd.DataFrame(results)


In [ ]:
tctc_tscs_ablation = run_tctc_tscs_ablation(rhos=(0.1, 0.3), epochs=20, T=4000, n_ts=10, n_f=5, n_rolling=10)
display(tctc_tscs_ablation.style.format(precision=3))
display(tctc_tscs_ablation.pivot(index='effect', columns='rho', values='test_corr_optimal').style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))


## Sweep: number of features and training epochs

This section now uses the project generator directly and sweeps `n_f` and `epochs` for a fixed generator effect.
The default hard case is `TSCS_shift`.


In [ ]:
def run_tctc_nf_epoch_sweep(
    effect='TSCS_shift',
    rho=0.1,
    T=4000,
    n_ts=10,
    n_rolling=10,
    feature_list=(1, 2, 5),
    epoch_list=(10, 20, 40),
    batch_size=256,
    seed=0,
):
    mask_c = build_attention_mask(causal_mask, n_rolling, device=device)
    results = []

    for n_f in feature_list:
        for epochs in epoch_list:
            X, y, y_opt, _ = generate_project_generator_dataset(
                effect=effect,
                T=T,
                n_ts=n_ts,
                n_f=n_f,
                rho=rho,
                n_rolling=n_rolling,
                seed=seed,
            )

            train_loader, test_loader = build_loaders_from_tensors(
                X, y, n_rolling=n_rolling, batch_size=batch_size
            )

            model = models.CustomBiDimensionalTransformer(
                n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
                dropout=0.1, d_model=64, dim_feedforward=256, sparsify=None,
                roll_y=True, embeddings='both',
            ).to(device)

            wrapper = torch_benchmarks.TorchWrapper(
                model,
                optimizer=torch.optim.AdamW(model.parameters(), lr=0.001),
                loss_fn=nn.MSELoss(),
                device=device,
            )

            wrapper.fit(train_loader, test=test_loader, epochs=epochs, plot=False, verbose=0)

            results.append({
                'effect': effect,
                'n_f': n_f,
                'epochs': epochs,
                'train_corr_optimal': corr_to_optimal(wrapper, train_loader, y_opt),
                'test_corr_optimal': corr_to_optimal(wrapper, test_loader, y_opt),
                'last_train_batch_corr': wrapper.train_corr[-1],
                'last_test_batch_corr': wrapper.test_corr[-1],
            })

    return pd.DataFrame(results)


tctc_nf_epoch_sweep = run_tctc_nf_epoch_sweep(
    effect='TSCS_shift',
    rho=0.1,
    feature_list=(1, 2, 5),
    epoch_list=(10, 20, 40),
)


In [ ]:
display(tctc_nf_epoch_sweep.style.format(precision=3))
display(tctc_nf_epoch_sweep.pivot(index='n_f', columns='epochs', values='test_corr_optimal').style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))


## Joint spatiotemporal attention vs TCTC

This section now uses the project generator directly.
The goal is to compare the factorized `TCTC` model to the joint attention model on the same generator effects used elsewhere in the notebook.


In [ ]:
def run_joint_vs_tctc_experiments(
    T=4000,
    n_ts=10,
    n_f=5,
    n_rolling=10,
    batch_size=256,
    seed=0,
):
    cases = [
        {'case': 'TS_shift_rho_0.30', 'effect': 'TS_shift', 'rho': 0.30},
        {'case': 'CS_shift_rho_0.30', 'effect': 'CS_shift', 'rho': 0.30},
        {'case': 'TSCS_shift_rho_0.10', 'effect': 'TSCS_shift', 'rho': 0.10},
        {'case': 'TSCS_shift_rho_0.30', 'effect': 'TSCS_shift', 'rho': 0.30},
    ]

    model_builders = {
        'TCTC': lambda mask_c: models.CustomBiDimensionalTransformer(
            n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
            dropout=0.05, d_model=64, dim_feedforward=256, sparsify=None,
            roll_y=True, embeddings='both',
        ).to(device),
        'Joint_ST': lambda mask_c: models.JointSpatioTemporalTransformer(
            n_ts, n_f, n_rolling, nhead=8, num_layers=2,
            dropout=0.05, d_model=64, dim_feedforward=256, sparsify=None,
            roll_y=True, embeddings='both',
        ).to(device),
    }

    mask_c = build_attention_mask(causal_mask, n_rolling, device=device)
    results = []

    for case in cases:
        X, y, y_opt, _ = generate_project_generator_dataset(
            effect=case['effect'],
            T=T,
            n_ts=n_ts,
            n_f=n_f,
            rho=case['rho'],
            n_rolling=n_rolling,
            seed=seed,
        )

        train_loader, test_loader = build_loaders_from_tensors(
            X, y, n_rolling=n_rolling, batch_size=batch_size
        )

        for model_name, build_model in model_builders.items():
            model = build_model(mask_c)
            wrapper = torch_benchmarks.TorchWrapper(
                model,
                optimizer=torch.optim.AdamW(model.parameters(), lr=0.001),
                loss_fn=nn.MSELoss(),
                device=device,
            )
            wrapper.fit(train_loader, test=test_loader, epochs=40, plot=False, verbose=0)

            results.append({
                'case': case['case'],
                'effect': case['effect'],
                'rho': case['rho'],
                'model': model_name,
                'train_corr_optimal': corr_to_optimal(wrapper, train_loader, y_opt),
                'test_corr_optimal': corr_to_optimal(wrapper, test_loader, y_opt),
                'last_train_batch_corr': wrapper.train_corr[-1],
                'last_test_batch_corr': wrapper.test_corr[-1],
            })

    return pd.DataFrame(results)


joint_vs_tctc_results = run_joint_vs_tctc_experiments()


In [ ]:
display(joint_vs_tctc_results.style.format(precision=3))
display(
    joint_vs_tctc_results
    .pivot(index='case', columns='model', values='test_corr_optimal')
    .style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)
    .format(precision=3)
)
